In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
ee.Authenticate()
ee.Initialize(project='coringa-mangrove')

In [ ]:
# Coringa Wildlife Sanctuary bounding box (paper uses ~16.7–17.0°N, 82.2–82.4°E)
CORINGA = ee.Geometry.Rectangle([82.13, 16.42, 82.23, 17.0])

# Date range matching paper's 2022 Sentinel-2 data
START = '2022-09-01'
END = '2022-12-31'

# RF params from paper
RF_TREES = 100
RF_VARS = 5
TEST_RATIO = 0.3

# Customized Thresholds (CT) from Table 2
CT = {
    'NDVI':  (0.4, None),    # >= 0.4
    'NDMI':  (-0.34, -0.14), # -0.34 to -0.14
    'MNDWI': (-0.35, -0.14), # -0.35 to -0.14
    'GCVI':  (3.7, None),    # >= 3.7
    'SR':    (6.5, 11),      # 6.5 to 11
}

In [ ]:
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(CORINGA)
      .filterDate(START, END)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .median()
      .clip(CORINGA))

print('Sentinel-2 bands:', s2.bandNames().getInfo())

Sentinel-2 bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


In [ ]:
# --- Compute 5 spectral indices ---
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndmi = s2.normalizedDifference(['B8', 'B11']).rename('NDMI')
mndwi = s2.normalizedDifference(['B3', 'B11']).rename('MNDWI')
gcvi = s2.expression('B8 / B3 - 1', {'B8': s2.select('B8'), 'B3': s2.select('B3')}).rename('GCVI')
sr = s2.expression('B8 / B4', {'B8': s2.select('B8'), 'B4': s2.select('B4')}).rename('SR')
indices = ee.Image.cat([ndvi, ndmi, mndwi, gcvi, sr])

# --- Apply CT masks ---
def make_ct_mask(band, lo, hi):
    if lo is not None and hi is not None:
        return band.gte(lo).And(band.lte(hi))
    elif lo is not None:
        return band.gte(lo)
    elif hi is not None:
        return band.lte(hi)
    return ee.Image.constant(1)

ct_masks = []
for name, (lo, hi) in CT.items():
    ct_masks.append(make_ct_mask(indices.select(name), lo, hi).rename(f'CT_{name}'))
ct_masks = ee.Image.cat(ct_masks)

# --- Stack all features ---
base = s2.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])
features = ee.Image.cat([base, indices, ct_masks])

print('Feature bands:', features.bandNames().getInfo())

Feature bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDMI', 'MNDWI', 'GCVI', 'SR', 'CT_NDVI', 'CT_NDMI', 'CT_MNDWI', 'CT_GCVI', 'CT_SR']


In [ ]:
!pip install earthengine-api geemap scikit-learn pandas numpy matplotlib seaborn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.0 MB/s eta 0:00:00


In [ ]:
# Try the data catalog search
try:
    catalog = ee.data.getCatalog()
    for item in catalog:
        if 'mangrove' in str(item).lower() or 'gmw' in str(item).lower():
            print(item)
except:
    pass

# Known working GMW paths to try
paths = [
    'projects/sat-io/open-datasets/global-mangrove-watch/gmw_2020',
    'projects/sat-io/open-datasets/global-mangrove-watch/gmw_2010',
    'UMD/global_mangrove_watch/gmw_2020_v2',
    'UMD/global_mangrove_watch/gmw_2020',
]
for p in paths:
    try:
        fc = ee.FeatureCollection(p).limit(1)
        print(f'✅ FeatureCollection: {p}')
    except:
        try:
            ic = ee.ImageCollection(p).limit(1)
            print(f'✅ ImageCollection: {p}')
        except:
            print(f'❌ {p}')

✅ FeatureCollection: projects/sat-io/open-datasets/global-mangrove-watch/gmw_2020
✅ FeatureCollection: projects/sat-io/open-datasets/global-mangrove-watch/gmw_2010
✅ FeatureCollection: UMD/global_mangrove_watch/gmw_2020_v2
✅ FeatureCollection: UMD/global_mangrove_watch/gmw_2020


In [ ]:
# --- Generate training labels using CT rule (mangrove where ALL CT conditions met) ---
ct_all = ee.Image.cat([
    indices.select('NDVI').gte(0.4),
    indices.select('NDMI').gte(-0.34).And(indices.select('NDMI').lte(-0.14)),
    indices.select('MNDWI').gte(-0.35).And(indices.select('MNDWI').lte(-0.14)),
    indices.select('GCVI').gte(3.7),
    indices.select('SR').gte(6.5).And(indices.select('SR').lte(11)),
])

# Mangrove = all 5 CT conditions met (1), else non-mangrove (0)
label_image = ct_all.reduce(ee.Reducer.allNonZero()).rename('label')

# --- Sample training points ---
samples = label_image.addBands(features).stratifiedSample(
    numPoints=1000,
    classBand='label',
    region=CORINGA,
    scale=10,
    geometries=False,
    seed=42
)

print('Samples collected:', samples.size().getInfo())

Samples collected: 1000


In [ ]:
# --- Prepare features & labels for sklearn ---
df = samples.select(features.bandNames().add('label')).getInfo()
data_list = []
for f in df['features']:
    props = f['properties']
    data_list.append(props)

df = pd.DataFrame(data_list)

# Separate features and labels
feature_names = features.bandNames().getInfo()
X = df[feature_names]
y = df['label']

print(f'X shape: {X.shape}')
print(f'Class distribution:\n{y.value_counts()}')# NDVI >= 0.4 as mangrove label proxy (best single index from paper)
label_image = indices.select('NDVI').gte(0.4).rename('label')

samples = label_image.addBands(features).stratifiedSample(
    numPoints=2000,
    classBand='label',
    region=CORINGA,
    scale=10,
    geometries=False,
    seed=42
)

df = pd.DataFrame([f['properties'] for f in samples.getInfo()['features']])
feature_names = features.bandNames().getInfo()
X = df[feature_names]
y = df['label']

print(f'X: {X.shape}, class 0: {(y==0).sum()}, class 1: {(y==1).sum()}')

X shape: (1000, 16)
Class distribution:
label
0    1000
Name: count, dtype: int64
X: (4000, 16), class 0: 2000, class 1: 2000


In [ ]:
# NDVI >= 0.4 as mangrove label proxy (best single index from paper)
label_image = indices.select('NDVI').gte(0.4).rename('label')

samples = label_image.addBands(features).stratifiedSample(
    numPoints=2000,
    classBand='label',
    region=CORINGA,
    scale=10,
    geometries=False,
    seed=42
)

df = pd.DataFrame([f['properties'] for f in samples.getInfo()['features']])
feature_names = features.bandNames().getInfo()
X = df[feature_names]
y = df['label']

print(f'X: {X.shape}, class 0: {(y==0).sum()}, class 1: {(y==1).sum()}')

X: (4000, 16), class 0: 2000, class 1: 2000


In [ ]:
# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# --- Train RF (paper params: 100 trees, 5 vars) ---
rf = RandomForestClassifier(
    n_estimators=100,
    max_features=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# --- Evaluate ---
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'\nTest Accuracy: {acc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Non-Mangrove', 'Mangrove']))

# --- Feature importance ---
importances = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print('\nTop 10 Feature Importances:')
print(importances.head(10))


Test Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

Non-Mangrove       1.00      1.00      1.00       600
    Mangrove       1.00      1.00      1.00       600

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200


Top 10 Feature Importances:
    feature  importance
11  CT_NDVI    0.296185
10       SR    0.293142
6      NDVI    0.221795
9      GCVI    0.098120
8     MNDWI    0.029379
4       B11    0.021807
3        B8    0.019574
5       B12    0.013942
2        B4    0.001669
15    CT_SR    0.001307


In [ ]:
#Load ESA WorldCover 2021 (10m) as mangrove/non-mangrove labels
esa = ee.Image('ESA/WorldCover/v200/2021').select('Map')
# Mangrove = class 95 in WorldCover
label_image = esa.eq(95).rename('label')

In [ ]:
# Keep only labeled pixels (nodata/coastline excluded)
valid_mask = esa.gt(0)
label_image = label_image.updateMask(valid_mask)

# --- Sample ---
samples = label_image.addBands(features).stratifiedSample(
    numPoints=2000,
    classBand='label',
    region=CORINGA,
    scale=10,
    geometries=False,
    seed=42
)

df = pd.DataFrame([f['properties'] for f in samples.getInfo()['features']])
feature_names = features.bandNames().getInfo()
X = df[feature_names]
y = df['label']

print(f'X: {X.shape}')
print(f'Class distribution:\n{y.value_counts()}')

X: (4000, 16)
Class distribution:
label
0    2000
1    2000
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# --- RF with paper params ---
rf = RandomForestClassifier(n_estimators=100, max_features=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Non-Mangrove', 'Mangrove']))

# --- Feature importance ---
importances = pd.DataFrame({
    'feature': feature_names, 'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print('\nTop 10 Feature Importances:')
print(importances.head(10))

# --- IoU for RF ---
def compute_iou(y_true, y_pred):
    intersection = (y_true.values.astype(bool) & y_pred.astype(bool)).sum()
    union = (y_true.values.astype(bool) | y_pred.astype(bool)).sum()
    return intersection / (union + 1e-7)

iou_rf = compute_iou(y_test, y_pred)
print(f'\nRF+CT IoU: {iou_rf:.4f}')

Test Accuracy: 0.9475

Classification Report:
              precision    recall  f1-score   support

Non-Mangrove       0.95      0.94      0.95       600
    Mangrove       0.94      0.95      0.95       600

    accuracy                           0.95      1200
   macro avg       0.95      0.95      0.95      1200
weighted avg       0.95      0.95      0.95      1200


Top 10 Feature Importances:
   feature  importance
7     NDMI    0.271461
8    MNDWI    0.143357
10      SR    0.092559
6     NDVI    0.091102
5      B12    0.088378
4      B11    0.068711
9     GCVI    0.056862
3       B8    0.051738
0       B2    0.044612
1       B3    0.036809


In [ ]:
# --- XGBoost on same 16 features ---
!pip install xgboost -q

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print(f'XGBoost Accuracy (16 features): {accuracy_score(y_test, y_pred_xgb):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_xgb, target_names=['Non-Mangrove', 'Mangrove']))

# Feature importance
importances_xgb = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False)
print('\nTop 10 Feature Importances:')
print(importances_xgb.head(10))

# --- IoU for fair comparison with DL ---
def compute_iou(y_true, y_pred):
    intersection = (y_true.values.astype(bool) & y_pred.astype(bool)).sum()
    union = (y_true.values.astype(bool) | y_pred.astype(bool)).sum()
    return intersection / (union + 1e-7)

iou_rf = compute_iou(y_test, y_pred)
iou_xgb = compute_iou(y_test, y_pred_xgb)
print(f'\n--- IoU Comparison ---')
print(f'RF+CT IoU:    {iou_rf:.4f}')
print(f'XGBoost IoU:  {iou_xgb:.4f}')
print(f'DeepLabV3+:   0.6454')

In [ ]:
# --- XGBoost on same 11 features as DL model (fair comparison) ---
dl_features = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDMI', 'MNDWI', 'GCVI', 'SR']

X_dl = df[dl_features]
y_dl = df['label']

X_dl_train, X_dl_test, y_dl_train, y_dl_test = train_test_split(
    X_dl, y_dl, test_size=0.3, random_state=42, stratify=y_dl
)

xgb_dl = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)
xgb_dl.fit(X_dl_train, y_dl_train)
y_pred_dl = xgb_dl.predict(X_dl_test)

print(f'XGBoost Accuracy (11 features, DL-comparable): {accuracy_score(y_dl_test, y_pred_dl):.4f}')
print('\nClassification Report:')
print(classification_report(y_dl_test, y_pred_dl, target_names=['Non-Mangrove', 'Mangrove']))